# 05. Content Feature Engineering

이 노트북의 목적은 `03_movie_metadata_unification.ipynb`에서 생성한 `movie_metadata_unified_v2.csv`를 `02_preprocessing_policy.ipynb`의 3주 관측창 시청이력에 붙여, 구독 이벤트 단위의 콘텐츠 성향 파생변수를 생성하는 것이다.

이 버전은 `04_usage_feature_engineering.ipynb`의 watch gap feature 패치가 반영된 산출물, 즉 `modeling_feature_table_usage.csv`가 정상 생성되어 있다는 전제에서 실행한다. 04번의 원본 오류는 05번 설계 자체의 문제는 아니지만, 05번은 04번의 최종 산출물을 입력으로 받기 때문에 패치본 04번 이후 실행하는 것을 기준으로 한다.

핵심 원칙은 다음과 같다.

1. 분석 단위는 `membership_row_id`이다.
2. 시청이력은 02번에서 만든 고객별 `reg_date` 기준 day 0~20 관측창만 사용한다.
3. 영화 메타데이터는 03번의 `movie_metadata_unified_v2.csv`를 사용한다.
4. KOBIS 저신뢰 매칭은 CSV에는 남겨두되, `use_for_content_features == 0`이면 콘텐츠 피처 계산에서 제외한다.
5. 장르는 대표 장르 순서가 없으므로 `top1` 방식으로 강제 선택하지 않는다.
6. 장르는 콘텐츠 성향 태그로 보고, 멀티핫 방식의 `tag_ratio_*`를 중심으로 생성한다.
7. 균등 배분 방식의 `alloc_ratio_*`는 보조 검증 변수로 생성한다.
8. 콘텐츠 피처는 이탈의 단독 원인으로 단정하지 않고, 100원딜/요금제/시청패턴 세그먼트를 설명하는 보조 변수로 사용한다.


## 5-1. 라이브러리 로딩


In [1]:
from pathlib import Path
import json
import math
import re
from collections import Counter

import numpy as np
import pandas as pd


## 5-2. 경로 설정

이 노트북은 `park.ingyeom` 폴더 안에서 실행하는 것을 기준으로 한다. 02번, 03번, 04번 노트북이 먼저 실행되어 있어야 한다.


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / ".git").exists() and (candidate / "_data").exists():
            return candidate

    for candidate in candidates:
        if candidate.name == "park.ingyeom" and (candidate.parent / "_data").exists():
            return candidate.parent

    raise FileNotFoundError("저장소 루트를 찾지 못했습니다.")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "_data"

WORK_ROOT = PROJECT_ROOT / "park.ingyeom"
REPORTS_DIR = WORK_ROOT / "reports"

INPUT_02_DATA_DIR = REPORTS_DIR / "data" / "02_preprocessing_policy"
INPUT_03_DATA_DIR = REPORTS_DIR / "data" / "03_movie_metadata_unification"
INPUT_04_DATA_DIR = REPORTS_DIR / "data" / "04_usage_feature_engineering"

OUTPUT_DATA_DIR = REPORTS_DIR / "data" / "05_content_feature_engineering"
OUTPUT_TABLE_DIR = REPORTS_DIR / "tables" / "05_content_feature_engineering"

# 기존 셀들의 저장 코드를 최소 수정하기 위해 TABLES_DIR 별칭을 유지한다.
TABLES_DIR = OUTPUT_TABLE_DIR

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("REPORTS_DIR:", REPORTS_DIR)
print("INPUT_02_DATA_DIR:", INPUT_02_DATA_DIR)
print("INPUT_03_DATA_DIR:", INPUT_03_DATA_DIR)
print("INPUT_04_DATA_DIR:", INPUT_04_DATA_DIR)
print("OUTPUT_DATA_DIR:", OUTPUT_DATA_DIR)
print("OUTPUT_TABLE_DIR:", OUTPUT_TABLE_DIR)


PROJECT_ROOT: c:\Code\ott-churn-prediction
DATA_ROOT: c:\Code\ott-churn-prediction\_data
REPORTS_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports
INPUT_02_DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\02_preprocessing_policy
INPUT_03_DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\03_movie_metadata_unification
INPUT_04_DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\04_usage_feature_engineering
OUTPUT_DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\05_content_feature_engineering
OUTPUT_TABLE_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\tables\05_content_feature_engineering


## 5-3. 입력 파일 로딩

05번은 원본 파일을 다시 전처리하지 않는다. 반드시 앞 단계 산출물을 입력으로 사용한다.

필수 입력은 다음과 같다.

- `_data/02_interim/view_history_observation_window.csv`
- `_data/02_interim/movie_metadata_unified_v2.csv`
- `_data/03_processed/modeling_feature_table_usage.csv`

`modeling_feature_table_usage.csv`는 04번 패치본 기준으로 생성되어야 한다. 특히 watch gap feature가 `groupby().apply(...).unstack().reset_index()` 방식으로 wide table로 펴진 결과여야 한다.


In [3]:
INPUT_FILES = {
    "obs_view": INPUT_02_DATA_DIR / "view_history_observation_window.csv",
    "movie_metadata_v2": INPUT_03_DATA_DIR / "movie_metadata_unified_v2.csv",
    "modeling_usage": INPUT_04_DATA_DIR / "modeling_feature_table_usage.csv",
}

missing = [str(p) for p in INPUT_FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        "05번 실행 전 02, 03, 04번 노트북 산출물이 필요합니다. Missing: " + str(missing)
    )

obs_view = pd.read_csv(INPUT_FILES["obs_view"])
movie_meta = pd.read_csv(INPUT_FILES["movie_metadata_v2"])
modeling_usage = pd.read_csv(INPUT_FILES["modeling_usage"])

if "watch_time(min)" in obs_view.columns and "watch_time" not in obs_view.columns:
    obs_view = obs_view.rename(columns={"watch_time(min)": "watch_time"})

if "watch_day" in obs_view.columns:
    obs_view["watch_day"] = pd.to_datetime(obs_view["watch_day"], errors="coerce")

file_summary = pd.DataFrame([
    {"name": "view_history_observation_window", "path": str(INPUT_FILES["obs_view"]), "rows": len(obs_view), "cols": obs_view.shape[1]},
    {"name": "movie_metadata_unified_v2", "path": str(INPUT_FILES["movie_metadata_v2"]), "rows": len(movie_meta), "cols": movie_meta.shape[1]},
    {"name": "modeling_feature_table_usage", "path": str(INPUT_FILES["modeling_usage"]), "rows": len(modeling_usage), "cols": modeling_usage.shape[1]},
])
file_summary.to_csv(TABLES_DIR / "05_content_feature_input_file_summary.csv", index=False, encoding="utf-8-sig")
display(file_summary)


,name,path,rows,cols
0,view_history_observation_window,c:\Code\ott-churn-prediction\park.ingyeom\repo...,85066,14
1,movie_metadata_unified_v2,c:\Code\ott-churn-prediction\park.ingyeom\repo...,14018,70
2,modeling_feature_table_usage,c:\Code\ott-churn-prediction\park.ingyeom\repo...,14922,96


## 5-4. 필수 컬럼 검산


In [4]:
required_obs_cols = {'membership_row_id', 'MOVIE_NUM', 'watch_time'}
required_meta_cols = {
    'MOVIE_NUM', 'movie_title', 'metadata_source', 'metadata_quality',
    'use_for_content_features', 'metadata_covered_flag',
    'unified_genres', 'unified_countries', 'unified_age_rating',
    'unified_runtime_min', 'unified_release_year',
}
required_usage_cols = {'membership_row_id', 'is_repurchase', 'is_100won', 'max_screen'}

missing_obs_cols = sorted(required_obs_cols - set(obs_view.columns))
missing_meta_cols = sorted(required_meta_cols - set(movie_meta.columns))
missing_usage_cols = sorted(required_usage_cols - set(modeling_usage.columns))

if missing_obs_cols:
    raise ValueError(f'obs_view 필수 컬럼 누락: {missing_obs_cols}')
if missing_meta_cols:
    raise ValueError(f'movie_metadata_v2 필수 컬럼 누락: {missing_meta_cols}')
if missing_usage_cols:
    raise ValueError(f'modeling_usage 필수 컬럼 누락: {missing_usage_cols}')

input_check = pd.DataFrame([
    {'check': 'obs_view_membership_row_id_unique', 'value': int(obs_view['membership_row_id'].nunique())},
    {'check': 'obs_view_rows', 'value': int(len(obs_view))},
    {'check': 'obs_view_unique_movies', 'value': int(obs_view['MOVIE_NUM'].nunique())},
    {'check': 'movie_meta_rows', 'value': int(len(movie_meta))},
    {'check': 'movie_meta_unique_movies', 'value': int(movie_meta['MOVIE_NUM'].nunique())},
    {'check': 'modeling_usage_rows', 'value': int(len(modeling_usage))},
    {'check': 'modeling_usage_unique_membership_row_id', 'value': int(modeling_usage['membership_row_id'].nunique())},
])
input_check.to_csv(TABLES_DIR / '05_content_feature_input_key_check.csv', index=False, encoding='utf-8-sig')
display(input_check)


,check,value
0,obs_view_membership_row_id_unique,12302
1,obs_view_rows,85066
2,obs_view_unique_movies,4765
3,movie_meta_rows,14018
4,movie_meta_unique_movies,14018
5,modeling_usage_rows,14922
6,modeling_usage_unique_membership_row_id,14922


## 5-5. 시청이력과 영화 메타데이터 연결

`View_History.MOVIE_NUM`과 `movie_metadata_unified_v2.MOVIE_NUM`을 기준으로 연결한다.

메타데이터가 없거나 저신뢰로 판단된 영화는 join 결과에는 남긴다. 다만 콘텐츠 성향 피처 계산에서는 `use_for_content_features == 1`인 행만 사용한다.


In [5]:
obs = obs_view.copy()
meta = movie_meta.copy()

obs['MOVIE_NUM'] = pd.to_numeric(obs['MOVIE_NUM'], errors='coerce').astype('Int64')
meta['MOVIE_NUM'] = pd.to_numeric(meta['MOVIE_NUM'], errors='coerce').astype('Int64')
obs['watch_time'] = pd.to_numeric(obs['watch_time'], errors='coerce').fillna(0)

view_meta = obs.merge(meta, on='MOVIE_NUM', how='left', suffixes=('', '_movie'))

# join되지 않은 영화는 missing 처리한다.
view_meta['metadata_covered_flag'] = view_meta['metadata_covered_flag'].fillna(0).astype(int)
view_meta['use_for_content_features'] = view_meta['use_for_content_features'].fillna(0).astype(int)
view_meta['metadata_source'] = view_meta['metadata_source'].fillna('missing_after_join')
view_meta['metadata_quality'] = view_meta['metadata_quality'].fillna('E_missing_after_join')

join_summary = pd.DataFrame([
    {'metric': 'obs_view_rows', 'value': int(len(obs))},
    {'metric': 'view_meta_rows', 'value': int(len(view_meta))},
    {'metric': 'unique_movies_in_obs', 'value': int(obs['MOVIE_NUM'].nunique())},
    {'metric': 'unique_movies_joined_usable', 'value': int(view_meta.loc[view_meta['use_for_content_features'] == 1, 'MOVIE_NUM'].nunique())},
    {'metric': 'watch_time_total', 'value': float(view_meta['watch_time'].sum())},
    {'metric': 'watch_time_metadata_covered', 'value': float(view_meta.loc[view_meta['metadata_covered_flag'] == 1, 'watch_time'].sum())},
    {'metric': 'watch_time_usable_for_content_features', 'value': float(view_meta.loc[view_meta['use_for_content_features'] == 1, 'watch_time'].sum())},
])
join_summary.to_csv(TABLES_DIR / '05_content_feature_view_metadata_join_summary.csv', index=False, encoding='utf-8-sig')
display(join_summary)


,metric,value
0,obs_view_rows,85066.0
1,view_meta_rows,85066.0
2,unique_movies_in_obs,4765.0
3,unique_movies_joined_usable,4755.0
4,watch_time_total,3800349.0
5,watch_time_metadata_covered,3799654.0
6,watch_time_usable_for_content_features,3799654.0


## 5-6. 메타데이터 커버리지 피처

콘텐츠 피처는 영화 메타데이터가 붙은 시청기록에만 의존한다. 따라서 유저별로 메타데이터 커버율 자체를 피처로 남긴다.


In [6]:
def safe_divide(numer, denom):
    return np.where(denom > 0, numer / denom, 0)

coverage_features = view_meta.groupby('membership_row_id').agg(
    total_watch_time_for_content=('watch_time', 'sum'),
    content_total_sessions=('watch_time', 'count'),
    content_unique_movies=('MOVIE_NUM', 'nunique'),
    metadata_covered_watch_time=('watch_time', lambda s: s[view_meta.loc[s.index, 'metadata_covered_flag'] == 1].sum()),
    usable_metadata_watch_time=('watch_time', lambda s: s[view_meta.loc[s.index, 'use_for_content_features'] == 1].sum()),
).reset_index()

coverage_features['metadata_covered_watch_ratio'] = safe_divide(
    coverage_features['metadata_covered_watch_time'],
    coverage_features['total_watch_time_for_content'],
)
coverage_features['usable_metadata_watch_ratio'] = safe_divide(
    coverage_features['usable_metadata_watch_time'],
    coverage_features['total_watch_time_for_content'],
)
coverage_features['metadata_missing_watch_time'] = (
    coverage_features['total_watch_time_for_content'] - coverage_features['metadata_covered_watch_time']
)
coverage_features['metadata_missing_watch_ratio'] = safe_divide(
    coverage_features['metadata_missing_watch_time'],
    coverage_features['total_watch_time_for_content'],
)

coverage_features.head()


,membership_row_id,total_watch_time_for_content,content_total_sessions,content_unique_movies,metadata_covered_watch_time,usable_metadata_watch_time,metadata_covered_watch_ratio,usable_metadata_watch_ratio,metadata_missing_watch_time,metadata_missing_watch_ratio
0,0,518,13,10,518,518,1.0,1.0,0,0.0
1,1,4,3,3,4,4,1.0,1.0,0,0.0
2,4,129,2,2,129,129,1.0,1.0,0,0.0
3,7,99,2,1,99,99,1.0,1.0,0,0.0
4,9,276,22,20,276,276,1.0,1.0,0,0.0


## 5-7. 문자열 분리와 컬럼명 정리 함수


In [7]:
def split_multi(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() == 'nan':
        return []
    parts = re.split(r'[|,]+', text)
    return [p.strip() for p in parts if p.strip()]


def safe_col_name(value: str) -> str:
    text = str(value).strip()
    text = text.replace('/', '_')
    text = text.replace(' ', '_')
    text = text.replace('-', '_')
    text = re.sub(r'[^0-9A-Za-z가-힣_]+', '', text)
    text = re.sub(r'_+', '_', text).strip('_')
    return text


def entropy_from_values(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[arr > 0]
    if arr.size == 0:
        return 0.0
    p = arr / arr.sum()
    return float(-(p * np.log(p)).sum())


def normalized_entropy_from_values(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[arr > 0]
    if arr.size <= 1:
        return 0.0
    ent = entropy_from_values(arr)
    return float(ent / np.log(arr.size))


## 5-8. 장르 피처 생성

장르 피처는 두 방식으로 만든다.

1. `tag_dur_*`, `tag_ratio_*`: multi-hot 태그 방식이다. 어떤 영화를 100분 봤고 장르가 `액션|스릴러/범죄`이면 액션 100분, 스릴러/범죄 100분으로 계산한다. 합이 실제 시청시간을 넘을 수 있지만, 콘텐츠 성향 태그로는 이 방식이 더 자연스럽다.
2. `alloc_dur_*`, `alloc_ratio_*`: 균등 배분 방식이다. 같은 예시에서 액션 50분, 스릴러/범죄 50분으로 계산한다. 합이 실제 시청시간과 맞기 때문에 다양성/entropy 계산에 적합하다.


In [8]:
usable = view_meta.loc[view_meta['use_for_content_features'] == 1].copy()
usable['genre_list'] = usable['unified_genres'].map(split_multi)

# multi-hot genre durations
rows = []
for row in usable[['membership_row_id', 'watch_time', 'genre_list']].itertuples(index=False):
    if not row.genre_list:
        continue
    for genre in row.genre_list:
        rows.append((row.membership_row_id, genre, row.watch_time, row.watch_time / len(row.genre_list)))

genre_long = pd.DataFrame(rows, columns=['membership_row_id', 'genre', 'tag_watch_time', 'alloc_watch_time'])

if len(genre_long) > 0:
    tag_genre = genre_long.pivot_table(
        index='membership_row_id', columns='genre', values='tag_watch_time', aggfunc='sum', fill_value=0
    )
    tag_genre.columns = [f'tag_dur_{safe_col_name(c)}' for c in tag_genre.columns]
    tag_genre = tag_genre.reset_index()

    alloc_genre = genre_long.pivot_table(
        index='membership_row_id', columns='genre', values='alloc_watch_time', aggfunc='sum', fill_value=0
    )
    alloc_genre.columns = [f'alloc_dur_{safe_col_name(c)}' for c in alloc_genre.columns]
    alloc_genre = alloc_genre.reset_index()
else:
    tag_genre = pd.DataFrame(columns=['membership_row_id'])
    alloc_genre = pd.DataFrame(columns=['membership_row_id'])

# genre count and entropy from allocation durations
genre_stats = genre_long.groupby('membership_row_id').agg(
    genre_tag_count=('genre', 'nunique')
).reset_index() if len(genre_long) > 0 else pd.DataFrame(columns=['membership_row_id', 'genre_tag_count'])

if len(genre_long) > 0:
    entropy_rows = []
    for mid, g in genre_long.groupby('membership_row_id'):
        alloc_by_genre = g.groupby('genre')['alloc_watch_time'].sum().values
        entropy_rows.append({
            'membership_row_id': mid,
            'genre_tag_entropy': entropy_from_values(alloc_by_genre),
            'genre_tag_entropy_norm': normalized_entropy_from_values(alloc_by_genre),
        })
    genre_entropy = pd.DataFrame(entropy_rows)
else:
    genre_entropy = pd.DataFrame(columns=['membership_row_id', 'genre_tag_entropy', 'genre_tag_entropy_norm'])

# top genre from tag duration
if len(genre_long) > 0:
    top_genre = (
        genre_long.groupby(['membership_row_id', 'genre'])['tag_watch_time']
        .sum()
        .reset_index()
        .sort_values(['membership_row_id', 'tag_watch_time', 'genre'], ascending=[True, False, True])
        .drop_duplicates('membership_row_id')
        .rename(columns={'genre': 'top_genre_tag', 'tag_watch_time': 'top_genre_tag_watch_time'})
    )
else:
    top_genre = pd.DataFrame(columns=['membership_row_id', 'top_genre_tag', 'top_genre_tag_watch_time'])

print('genre_long rows:', len(genre_long))
print('tag_genre shape:', tag_genre.shape)
print('alloc_genre shape:', alloc_genre.shape)


genre_long rows: 186779
tag_genre shape: (12302, 14)
alloc_genre shape: (12302, 14)


## 5-9. 관람등급 피처 생성


In [9]:
usable['rating_list'] = usable['unified_age_rating'].map(split_multi)

rating_rows = []
for row in usable[['membership_row_id', 'watch_time', 'rating_list']].itertuples(index=False):
    if not row.rating_list:
        continue
    for rating in row.rating_list:
        rating_rows.append((row.membership_row_id, rating, row.watch_time, row.watch_time / len(row.rating_list)))

rating_long = pd.DataFrame(rating_rows, columns=['membership_row_id', 'rating', 'tag_watch_time', 'alloc_watch_time'])

if len(rating_long) > 0:
    rating_features = rating_long.pivot_table(
        index='membership_row_id', columns='rating', values='alloc_watch_time', aggfunc='sum', fill_value=0
    )
    rating_features.columns = [f'rating_dur_{safe_col_name(c)}' for c in rating_features.columns]
    rating_features = rating_features.reset_index()
else:
    rating_features = pd.DataFrame(columns=['membership_row_id'])

rating_stats = rating_long.groupby('membership_row_id').agg(
    rating_tag_count=('rating', 'nunique')
).reset_index() if len(rating_long) > 0 else pd.DataFrame(columns=['membership_row_id', 'rating_tag_count'])

print('rating_long rows:', len(rating_long))
print('rating_features shape:', rating_features.shape)


rating_long rows: 79531
rating_features shape: (12089, 7)


## 5-10. 국가 피처 생성


In [10]:
usable['country_list'] = usable['unified_countries'].map(split_multi)

country_rows = []
for row in usable[['membership_row_id', 'watch_time', 'country_list']].itertuples(index=False):
    if not row.country_list:
        continue
    for country in row.country_list:
        country_rows.append((row.membership_row_id, country, row.watch_time, row.watch_time / len(row.country_list)))

country_long = pd.DataFrame(country_rows, columns=['membership_row_id', 'country', 'tag_watch_time', 'alloc_watch_time'])

# 국가 전체를 전부 column으로 만들면 과도해질 수 있으므로 주요 국가만 explicit feature로 만든다.
COUNTRY_MAP = {
    '한국': 'korean',
    '미국': 'us',
    '일본': 'japanese',
    '중국': 'chinese',
    '홍콩': 'hongkong',
    '영국': 'uk',
    '프랑스': 'french',
}

if len(country_long) > 0:
    country_long['country_group'] = country_long['country'].map(COUNTRY_MAP).fillna('other_foreign')
    country_features = country_long.pivot_table(
        index='membership_row_id', columns='country_group', values='alloc_watch_time', aggfunc='sum', fill_value=0
    )
    country_features.columns = [f'country_dur_{safe_col_name(c)}' for c in country_features.columns]
    country_features = country_features.reset_index()
else:
    country_features = pd.DataFrame(columns=['membership_row_id'])

country_stats = country_long.groupby('membership_row_id').agg(
    country_tag_count=('country', 'nunique')
).reset_index() if len(country_long) > 0 else pd.DataFrame(columns=['membership_row_id', 'country_tag_count'])

if len(country_long) > 0:
    country_entropy_rows = []
    for mid, g in country_long.groupby('membership_row_id'):
        alloc_by_country = g.groupby('country')['alloc_watch_time'].sum().values
        country_entropy_rows.append({
            'membership_row_id': mid,
            'country_entropy': entropy_from_values(alloc_by_country),
            'country_entropy_norm': normalized_entropy_from_values(alloc_by_country),
        })
    country_entropy = pd.DataFrame(country_entropy_rows)
else:
    country_entropy = pd.DataFrame(columns=['membership_row_id', 'country_entropy', 'country_entropy_norm'])

print('country_long rows:', len(country_long))
print('country_features shape:', country_features.shape)


country_long rows: 113459
country_features shape: (12267, 9)


## 5-11. 플래그형 콘텐츠 피처와 가중 평균 피처

영화 단위 통합 메타데이터에 있는 boolean flag와 수치형 메타데이터를 시청시간 가중 방식으로 유저 단위에 집계한다.


In [11]:
flag_cols = [
    'is_kids_animation', 'is_family_content', 'is_adult_content',
    'is_korean_content', 'is_us_content', 'is_japanese_content',
    'is_recent_content', 'is_old_content', 'is_long_movie', 'is_short_content',
]
flag_cols = [c for c in flag_cols if c in usable.columns]

for c in flag_cols:
    usable[c] = pd.to_numeric(usable[c], errors='coerce').fillna(0).astype(int)

flag_feature_parts = []
for c in flag_cols:
    tmp = usable.assign(weighted_flag=usable['watch_time'] * usable[c])
    agg = tmp.groupby('membership_row_id')['weighted_flag'].sum().reset_index()
    agg = agg.rename(columns={'weighted_flag': f'{c}_watch_time'})
    flag_feature_parts.append(agg)

flag_features = coverage_features[['membership_row_id']].copy()
for part in flag_feature_parts:
    flag_features = flag_features.merge(part, on='membership_row_id', how='left')

for c in [col for col in flag_features.columns if col.endswith('_watch_time')]:
    flag_features[c] = flag_features[c].fillna(0)
    ratio_col = c.replace('_watch_time', '_ratio')
    flag_features[ratio_col] = safe_divide(flag_features[c], coverage_features.set_index('membership_row_id').loc[flag_features['membership_row_id'], 'total_watch_time_for_content'].values)

# Weighted runtime and release year.
num = usable.copy()
num['unified_runtime_min'] = pd.to_numeric(num['unified_runtime_min'], errors='coerce')
num['unified_release_year'] = pd.to_numeric(num['unified_release_year'], errors='coerce')

weighted_rows = []
for mid, g in num.groupby('membership_row_id'):
    row = {'membership_row_id': mid}
    g_runtime = g.dropna(subset=['unified_runtime_min'])
    if len(g_runtime) and g_runtime['watch_time'].sum() > 0:
        row['avg_runtime_weighted'] = float((g_runtime['unified_runtime_min'] * g_runtime['watch_time']).sum() / g_runtime['watch_time'].sum())
    else:
        row['avg_runtime_weighted'] = 0.0

    g_year = g.dropna(subset=['unified_release_year'])
    if len(g_year) and g_year['watch_time'].sum() > 0:
        row['avg_release_year_weighted'] = float((g_year['unified_release_year'] * g_year['watch_time']).sum() / g_year['watch_time'].sum())
        row['avg_content_age_from_2021_weighted'] = float(((2021 - g_year['unified_release_year']) * g_year['watch_time']).sum() / g_year['watch_time'].sum())
    else:
        row['avg_release_year_weighted'] = 0.0
        row['avg_content_age_from_2021_weighted'] = 0.0
    weighted_rows.append(row)

weighted_numeric_features = pd.DataFrame(weighted_rows)
print('flag_features shape:', flag_features.shape)
print('weighted_numeric_features shape:', weighted_numeric_features.shape)


flag_features shape: (12302, 21)
weighted_numeric_features shape: (12302, 4)


## 5-12. 콘텐츠 피처 병합과 ratio 계산


In [12]:
content_features = coverage_features.copy()

for part in [
    tag_genre, alloc_genre, genre_stats, genre_entropy, top_genre,
    rating_features, rating_stats,
    country_features, country_stats, country_entropy,
    flag_features, weighted_numeric_features,
]:
    if len(part.columns) > 1:
        content_features = content_features.merge(part, on='membership_row_id', how='left')

# 결측 채우기
text_cols = ['top_genre_tag']
for c in text_cols:
    if c in content_features.columns:
        content_features[c] = content_features[c].fillna('no_usable_metadata')

for c in content_features.columns:
    if c not in ['membership_row_id', 'top_genre_tag']:
        content_features[c] = pd.to_numeric(content_features[c], errors='coerce').fillna(0)

# tag/alloc/rating/country duration columns를 ratio로 변환한다.
denominator = content_features['total_watch_time_for_content'].replace(0, np.nan)
for prefix in ['tag_dur_', 'alloc_dur_', 'rating_dur_', 'country_dur_']:
    dur_cols = [c for c in content_features.columns if c.startswith(prefix)]
    for c in dur_cols:
        ratio_prefix = prefix.replace('_dur_', '_ratio_')
        ratio_col = c.replace(prefix, ratio_prefix)
        content_features[ratio_col] = (content_features[c] / denominator).fillna(0)

# 주요 국가 ratio alias
alias_map = {
    'country_ratio_korean': 'korean_content_ratio_from_country',
    'country_ratio_us': 'us_content_ratio_from_country',
    'country_ratio_japanese': 'japanese_content_ratio_from_country',
    'country_ratio_other_foreign': 'other_foreign_content_ratio_from_country',
}
for src, dst in alias_map.items():
    if src in content_features.columns:
        content_features[dst] = content_features[src]

print('content_features shape:', content_features.shape)
content_features.head()


content_features shape: (12302, 126)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_34428\29850056.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  content_features[ratio_col] = (content_features[c] / denominator).fillna(0)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_34428\29850056.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  content_features[ratio_col] = (content_features[c] / denominator).fillna(0)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_34428\29850056.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually

,membership_row_id,total_watch_time_for_content,content_total_sessions,content_unique_movies,metadata_covered_watch_time,usable_metadata_watch_time,metadata_covered_watch_ratio,usable_metadata_watch_ratio,metadata_missing_watch_time,metadata_missing_watch_ratio,...,country_ratio_hongkong,country_ratio_japanese,country_ratio_korean,country_ratio_other_foreign,country_ratio_uk,country_ratio_us,korean_content_ratio_from_country,us_content_ratio_from_country,japanese_content_ratio_from_country,other_foreign_content_ratio_from_country
0,0,518,13,10,518,518,1.0,1.0,0,0.0,...,0.025097,0.193050,0.561776,0.001931,0.000000,0.218147,0.561776,0.218147,0.193050,0.001931
1,1,4,3,3,4,4,1.0,1.0,0,0.0,...,0.000000,0.000000,0.250000,0.500000,0.000000,0.250000,0.250000,0.250000,0.000000,0.500000
2,4,129,2,2,129,129,1.0,1.0,0,0.0,...,0.000000,0.000000,0.992248,0.000000,0.000000,0.000000,0.992248,0.000000,0.000000,0.000000
3,7,99,2,1,99,99,1.0,1.0,0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000
4,9,276,22,20,276,276,1.0,1.0,0,0.0,...,0.000000,0.060236,0.031250,0.024909,0.072917,0.775815,0.031250,0.775815,0.060236,0.024909


## 5-13. 콘텐츠 피처 품질 요약


In [13]:
content_numeric_cols = [c for c in content_features.columns if c not in ['membership_row_id', 'top_genre_tag']]
content_feature_summary = content_features[content_numeric_cols].describe().T.reset_index().rename(columns={'index': 'feature'})
content_feature_summary['missing_count'] = content_features[content_numeric_cols].isna().sum().values
content_feature_summary['zero_count'] = (content_features[content_numeric_cols] == 0).sum().values
content_feature_summary['zero_rate'] = content_feature_summary['zero_count'] / len(content_features)
content_feature_summary.to_csv(TABLES_DIR / '05_content_feature_numeric_summary.csv', index=False, encoding='utf-8-sig')

top_genre_counts = content_features['top_genre_tag'].value_counts(dropna=False).reset_index()
top_genre_counts.columns = ['top_genre_tag', 'count']
top_genre_counts.to_csv(TABLES_DIR / '05_content_feature_top_genre_counts.csv', index=False, encoding='utf-8-sig')

display(content_feature_summary.head(30))
display(top_genre_counts.head(20))


,feature,count,mean,std,min,25%,50%,75%,max,missing_count,zero_count,zero_rate
0,total_watch_time_for_content,12302.0,308.921232,320.195058,1.000000,96.0,218.000000,425.000000,4419.000000,0,0,0.000000
1,content_total_sessions,12302.0,6.914811,5.780675,1.000000,3.0,5.000000,9.000000,56.000000,0,0,0.000000
2,content_unique_movies,12302.0,4.963421,4.144566,1.000000,2.0,4.000000,7.000000,47.000000,0,0,0.000000
3,metadata_covered_watch_time,12302.0,308.864737,320.120527,1.000000,96.0,218.000000,425.000000,4419.000000,0,0,0.000000
4,usable_metadata_watch_time,12302.0,308.864737,320.120527,1.000000,96.0,218.000000,425.000000,4419.000000,0,0,0.000000
5,metadata_covered_watch_ratio,12302.0,0.999907,0.004758,0.608491,1.0,1.000000,1.000000,1.000000,0,0,0.000000
6,usable_metadata_watch_ratio,12302.0,0.999907,0.004758,0.608491,1.0,1.000000,1.000000,1.000000,0,0,0.000000
7,metadata_missing_watch_time,12302.0,0.056495,2.836202,0.000000,0.0,0.000000,0.000000,175.000000,0,12294,0.999350
8,metadata_missing_watch_ratio,12302.0,0.000093,0.004758,0.000000,0.0,0.000000,0.000000,0.391509,0,12294,0.999350
9,tag_dur_SF_판타지,12302.0,56.937246,125.552624,0.000000,0.0,0.000000,81.000000,1873.000000,0,6840,0.556007


,top_genre_tag,count
0,드라마,4164
1,액션,2131
2,스릴러/범죄,1343
3,SF/판타지,1192
4,애니메이션/키즈,869
5,코미디,595
6,모험/어드벤처,454
7,기타,405
8,가족,363
9,로맨스,274


## 5-14. 04번 사용 행동 테이블과 콘텐츠 피처 결합

최종 모델링 후보 테이블은 04번의 `modeling_feature_table_usage.csv`에 05번 콘텐츠 피처를 붙여 만든다.


In [14]:
modeling_with_content = modeling_usage.merge(content_features, on='membership_row_id', how='left')

# 시청이력이나 usable metadata가 없는 고객은 콘텐츠 수치형 피처를 0으로 채운다.
for c in content_features.columns:
    if c == 'membership_row_id':
        continue
    if c == 'top_genre_tag':
        modeling_with_content[c] = modeling_with_content[c].fillna('no_watch_or_no_usable_metadata')
    else:
        modeling_with_content[c] = pd.to_numeric(modeling_with_content[c], errors='coerce').fillna(0)

modeling_with_content['has_content_feature'] = (modeling_with_content['usable_metadata_watch_time'] > 0).astype(int)
modeling_with_content['no_content_feature_flag'] = (modeling_with_content['has_content_feature'] == 0).astype(int)

content_attach_check = pd.DataFrame([
    {'metric': 'modeling_usage_rows', 'value': int(len(modeling_usage))},
    {'metric': 'content_features_rows', 'value': int(len(content_features))},
    {'metric': 'modeling_with_content_rows', 'value': int(len(modeling_with_content))},
    {'metric': 'has_content_feature_rows', 'value': int(modeling_with_content['has_content_feature'].sum())},
    {'metric': 'no_content_feature_rows', 'value': int(modeling_with_content['no_content_feature_flag'].sum())},
])
content_attach_check.to_csv(TABLES_DIR / '05_content_feature_attach_check.csv', index=False, encoding='utf-8-sig')
display(content_attach_check)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_34428\835599424.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  modeling_with_content['has_content_feature'] = (modeling_with_content['usable_metadata_watch_time'] > 0).astype(int)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_34428\835599424.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  modeling_with_content['no_content_feature_flag'] = (modeling_with_content['has_content_feature'] == 0).astype(int)


,metric,value
0,modeling_usage_rows,14922
1,content_features_rows,12302
2,modeling_with_content_rows,14922
3,has_content_feature_rows,12302
4,no_content_feature_rows,2620


## 5-15. 가설형 콘텐츠 세그먼트 후보 생성

이 단계에서는 최종 결론을 내리지 않는다. 06번 유의성 검정에서 테스트할 후보 flag만 만든다.


In [15]:
def get_col(df, col, default=0):
    if col in df.columns:
        return df[col]
    return pd.Series(default, index=df.index)

kids_ratio = get_col(modeling_with_content, 'tag_ratio_애니메이션_키즈')
family_ratio = get_col(modeling_with_content, 'tag_ratio_가족')
all_age_ratio = get_col(modeling_with_content, 'rating_ratio_전체')
action_ratio = get_col(modeling_with_content, 'tag_ratio_액션')
sf_ratio = get_col(modeling_with_content, 'tag_ratio_SF_판타지')
thriller_ratio = get_col(modeling_with_content, 'tag_ratio_스릴러_범죄')
long_ratio = get_col(modeling_with_content, 'is_long_movie_ratio')
recent_ratio = get_col(modeling_with_content, 'is_recent_content_ratio')
w3_minus_w1 = get_col(modeling_with_content, 'w3_minus_w1_watch_time')
unique_days = get_col(modeling_with_content, 'unique_days')
is_user_verified_col = get_col(modeling_with_content, 'is_user_verified')
age_col = get_col(modeling_with_content, 'age')

modeling_with_content['family_content_affinity'] = (
    (kids_ratio >= 0.20) | (family_ratio >= 0.10) | (all_age_ratio >= 0.30)
).astype(int)

modeling_with_content['family_2screen_lifestyle'] = (
    (modeling_with_content['max_screen'] == 2)
    & (modeling_with_content['family_content_affinity'] == 1)
    & (unique_days >= 2)
).astype(int)

modeling_with_content['promo2_family_lifestyle_candidate'] = (
    (modeling_with_content['is_100won'] == 1)
    & (modeling_with_content['max_screen'] == 2)
    & (modeling_with_content['family_content_affinity'] == 1)
).astype(int)

modeling_with_content['action_sf_thriller_affinity'] = (
    ((action_ratio + sf_ratio + thriller_ratio) >= 0.40)
).astype(int)

modeling_with_content['premium_action_trial_risk'] = (
    (modeling_with_content['is_100won'] == 1)
    & (modeling_with_content['max_screen'] == 4)
    & (modeling_with_content['action_sf_thriller_affinity'] == 1)
    & ((long_ratio >= 0.30) | (w3_minus_w1 > 0))
).astype(int)

modeling_with_content['age40_2screen_anime_kids_candidate'] = (
    (modeling_with_content['is_100won'] == 1)
    & (modeling_with_content['max_screen'] == 2)
    & (is_user_verified_col == 1)
    & (age_col.between(40, 49))
    & ((kids_ratio >= 0.30) | (modeling_with_content['top_genre_tag'].astype(str) == '애니메이션/키즈'))
).astype(int)

modeling_with_content['promo4_long_recent_trial_candidate'] = (
    (modeling_with_content['is_100won'] == 1)
    & (modeling_with_content['max_screen'] == 4)
    & (long_ratio >= 0.30)
    & (recent_ratio >= 0.20)
).astype(int)

candidate_flags = [
    'family_content_affinity',
    'family_2screen_lifestyle',
    'promo2_family_lifestyle_candidate',
    'action_sf_thriller_affinity',
    'premium_action_trial_risk',
    'age40_2screen_anime_kids_candidate',
    'promo4_long_recent_trial_candidate',
]

segment_rate_rows = []
for col in candidate_flags:
    tmp = modeling_with_content.groupby(col)['is_repurchase'].agg(n='count', repurchase_rate='mean').reset_index()
    tmp.insert(0, 'segment_flag', col)
    tmp = tmp.rename(columns={col: 'flag_value'})
    segment_rate_rows.append(tmp)
segment_rate_table = pd.concat(segment_rate_rows, ignore_index=True)
segment_rate_table.to_csv(TABLES_DIR / '05_content_feature_candidate_segment_rates.csv', index=False, encoding='utf-8-sig')
display(segment_rate_table)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_34428\2314105828.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  modeling_with_content['family_content_affinity'] = (
C:\Users\Administrator\AppData\Local\Temp\ipykernel_34428\2314105828.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  modeling_with_content['family_2screen_lifestyle'] = (
C:\Users\Administrator\AppData\Local\Temp\ipykernel_34428\2314105828.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` m

,segment_flag,flag_value,n,repurchase_rate
0,family_content_affinity,0,11352,0.674154
1,family_content_affinity,1,3570,0.678711
2,family_2screen_lifestyle,0,14290,0.672498
3,family_2screen_lifestyle,1,632,0.737342
4,promo2_family_lifestyle_candidate,0,14564,0.673373
5,promo2_family_lifestyle_candidate,1,358,0.751397
6,action_sf_thriller_affinity,0,6672,0.671912
7,action_sf_thriller_affinity,1,8250,0.677939
8,premium_action_trial_risk,0,14195,0.686368
9,premium_action_trial_risk,1,727,0.458047


## 5-16. 주요 집단별 콘텐츠 요약

이 표는 06번의 정식 유의성 검정 전에 보는 탐색적 요약이다.


In [16]:
summary_cols = [
    'metadata_covered_watch_ratio', 'usable_metadata_watch_ratio',
    'tag_ratio_드라마', 'tag_ratio_액션', 'tag_ratio_스릴러_범죄', 'tag_ratio_SF_판타지',
    'tag_ratio_애니메이션_키즈', 'tag_ratio_가족',
    'rating_ratio_전체', 'rating_ratio_12세', 'rating_ratio_15세', 'rating_ratio_청불',
    'is_long_movie_ratio', 'is_recent_content_ratio', 'avg_runtime_weighted',
    'genre_tag_entropy_norm', 'country_entropy_norm',
]
summary_cols = [c for c in summary_cols if c in modeling_with_content.columns]

content_summary_by_100won = modeling_with_content.groupby('is_100won').agg(
    n=('membership_row_id', 'count'),
    repurchase_rate=('is_repurchase', 'mean'),
    **{f'avg_{c}': (c, 'mean') for c in summary_cols}
).reset_index()

content_summary_by_100won_screen = modeling_with_content.groupby(['is_100won', 'max_screen'], dropna=False).agg(
    n=('membership_row_id', 'count'),
    repurchase_rate=('is_repurchase', 'mean'),
    **{f'avg_{c}': (c, 'mean') for c in summary_cols}
).reset_index()

content_summary_by_100won.to_csv(TABLES_DIR / '05_content_feature_summary_by_100won.csv', index=False, encoding='utf-8-sig')
content_summary_by_100won_screen.to_csv(TABLES_DIR / '05_content_feature_summary_by_100won_maxscreen.csv', index=False, encoding='utf-8-sig')

display(content_summary_by_100won)
display(content_summary_by_100won_screen)


,is_100won,n,repurchase_rate,avg_metadata_covered_watch_ratio,avg_usable_metadata_watch_ratio,avg_tag_ratio_드라마,avg_tag_ratio_액션,avg_tag_ratio_스릴러_범죄,avg_tag_ratio_SF_판타지,avg_tag_ratio_애니메이션_키즈,avg_tag_ratio_가족,avg_rating_ratio_전체,avg_rating_ratio_12세,avg_rating_ratio_15세,avg_rating_ratio_청불,avg_is_long_movie_ratio,avg_is_recent_content_ratio,avg_avg_runtime_weighted,avg_genre_tag_entropy_norm,avg_country_entropy_norm
0,0,5939,0.741034,0.826912,0.826912,0.338231,0.298696,0.202396,0.149262,0.115176,0.079492,0.146859,0.188805,0.266212,0.132780,0.228619,0.321701,92.239624,0.616492,0.410471
1,1,8983,0.631749,0.822646,0.822646,0.339559,0.287537,0.197030,0.147023,0.118456,0.086117,0.149526,0.182515,0.266232,0.133525,0.226805,0.322868,91.342980,0.613932,0.404868


,is_100won,max_screen,n,repurchase_rate,avg_metadata_covered_watch_ratio,avg_usable_metadata_watch_ratio,avg_tag_ratio_드라마,avg_tag_ratio_액션,avg_tag_ratio_스릴러_범죄,avg_tag_ratio_SF_판타지,...,avg_tag_ratio_가족,avg_rating_ratio_전체,avg_rating_ratio_12세,avg_rating_ratio_15세,avg_rating_ratio_청불,avg_is_long_movie_ratio,avg_is_recent_content_ratio,avg_avg_runtime_weighted,avg_genre_tag_entropy_norm,avg_country_entropy_norm
0,0,1,3794,0.719294,0.829418,0.829418,0.334582,0.308231,0.206349,0.152469,...,0.078225,0.141660,0.194857,0.267800,0.131469,0.232437,0.324507,92.694307,0.619711,0.414001
1,0,2,1594,0.784191,0.813284,0.813284,0.343395,0.279562,0.198933,0.143134,...,0.081021,0.152199,0.178671,0.257325,0.135206,0.219711,0.320447,90.634555,0.603889,0.400600
2,0,4,551,0.765880,0.849082,0.849082,0.348415,0.288400,0.185194,0.144910,...,0.083792,0.167203,0.176449,0.280991,0.134788,0.228093,0.306008,93.752169,0.630787,0.414715
3,1,1,5413,0.658045,0.823910,0.823910,0.336470,0.288665,0.197573,0.150193,...,0.086259,0.148322,0.176228,0.270625,0.135250,0.223172,0.317689,91.425736,0.618291,0.410346
4,1,2,1528,0.740183,0.823953,0.823953,0.332113,0.288129,0.201937,0.140006,...,0.090302,0.152914,0.189228,0.260416,0.132593,0.231495,0.333662,91.444767,0.607788,0.404579
5,1,4,2042,0.480901,0.818315,0.818315,0.353318,0.284104,0.191919,0.143870,...,0.082608,0.150181,0.194159,0.258936,0.129648,0.232927,0.328520,91.047441,0.606977,0.390563


## 5-17. 산출물 저장


In [17]:
PATH_CONTENT_FEATURES = OUTPUT_DATA_DIR / "content_features.csv"
PATH_USER_CONTENT_FEATURES = PATH_CONTENT_FEATURES  # 기존 변수명 호환용 별칭
PATH_MODELING_WITH_CONTENT = OUTPUT_DATA_DIR / "modeling_feature_table_with_content.csv"
PATH_CONTENT_SUMMARY_JSON = OUTPUT_DATA_DIR / "content_feature_summary.json"

content_features.to_csv(PATH_CONTENT_FEATURES, index=False, encoding="utf-8-sig")
modeling_with_content.to_csv(PATH_MODELING_WITH_CONTENT, index=False, encoding="utf-8-sig")

summary = {
    "inputs": {k: str(v) for k, v in INPUT_FILES.items()},
    "outputs": {
        "content_features": str(PATH_CONTENT_FEATURES),
        "modeling_feature_table_with_content": str(PATH_MODELING_WITH_CONTENT),
        "content_feature_summary": str(PATH_CONTENT_SUMMARY_JSON),
    },
    "rows": {
        "obs_view": int(len(obs_view)),
        "movie_metadata_v2": int(len(movie_meta)),
        "modeling_usage": int(len(modeling_usage)),
        "view_meta": int(len(view_meta)),
        "content_features": int(len(content_features)),
        "modeling_with_content": int(len(modeling_with_content)),
    },
    "coverage": {
        "watch_time_total": float(view_meta["watch_time"].sum()),
        "watch_time_metadata_covered": float(view_meta.loc[view_meta["metadata_covered_flag"] == 1, "watch_time"].sum()),
        "watch_time_usable_for_content_features": float(view_meta.loc[view_meta["use_for_content_features"] == 1, "watch_time"].sum()),
    },
    "feature_counts": {
        "content_feature_columns": int(len(content_features.columns)),
        "modeling_with_content_columns": int(len(modeling_with_content.columns)),
    },
    "candidate_flags": candidate_flags,
}

with open(PATH_CONTENT_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("saved:", PATH_CONTENT_FEATURES)
print("saved:", PATH_MODELING_WITH_CONTENT)
print("saved:", PATH_CONTENT_SUMMARY_JSON)


saved: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\05_content_feature_engineering\content_features.csv
saved: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\05_content_feature_engineering\modeling_feature_table_with_content.csv
saved: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\05_content_feature_engineering\content_feature_summary.json


## 5-18. 최종 검산


In [18]:
expected_report_files = {
    "05_content_feature_input_file_summary.csv",
    "05_content_feature_input_key_check.csv",
    "05_content_feature_view_metadata_join_summary.csv",
    "05_content_feature_numeric_summary.csv",
    "05_content_feature_top_genre_counts.csv",
    "05_content_feature_attach_check.csv",
    "05_content_feature_candidate_segment_rates.csv",
    "05_content_feature_summary_by_100won.csv",
    "05_content_feature_summary_by_100won_maxscreen.csv",
    "05_content_feature_final_checks.csv",
}

actual_05_report_files = {p.name for p in TABLES_DIR.glob("05_*.csv")} | {"05_content_feature_final_checks.csv"}

final_check = pd.DataFrame([
    {"check": "project_root_is_repo_root", "value": str(PROJECT_ROOT), "pass": (PROJECT_ROOT / ".git").exists()},
    {"check": "data_root_is_repo_data", "value": str(DATA_ROOT), "pass": DATA_ROOT.exists()},
    {"check": "reports_dir_is_park_reports", "value": str(REPORTS_DIR), "pass": REPORTS_DIR == WORK_ROOT / "reports"},
    {"check": "input_02_data_dir_is_reports_data_02", "value": str(INPUT_02_DATA_DIR), "pass": INPUT_02_DATA_DIR == REPORTS_DIR / "data" / "02_preprocessing_policy"},
    {"check": "input_03_data_dir_is_reports_data_03", "value": str(INPUT_03_DATA_DIR), "pass": INPUT_03_DATA_DIR == REPORTS_DIR / "data" / "03_movie_metadata_unification"},
    {"check": "input_04_data_dir_is_reports_data_04", "value": str(INPUT_04_DATA_DIR), "pass": INPUT_04_DATA_DIR == REPORTS_DIR / "data" / "04_usage_feature_engineering"},
    {"check": "output_data_dir_under_reports_data_05", "value": str(OUTPUT_DATA_DIR), "pass": OUTPUT_DATA_DIR == REPORTS_DIR / "data" / "05_content_feature_engineering"},
    {"check": "output_table_dir_under_reports_tables_05", "value": str(OUTPUT_TABLE_DIR), "pass": OUTPUT_TABLE_DIR == REPORTS_DIR / "tables" / "05_content_feature_engineering"},
    {"check": "only_expected_05_report_csvs", "value": sorted(actual_05_report_files), "pass": actual_05_report_files == expected_report_files},
    {"check": "movie_meta_rows_is_14018", "value": int(len(movie_meta)), "pass": int(len(movie_meta)) == 14018},
    {"check": "modeling_usage_rows_is_14922", "value": int(len(modeling_usage)), "pass": int(len(modeling_usage)) == 14922},
    {"check": "modeling_with_content_rows_is_14922", "value": int(len(modeling_with_content)), "pass": int(len(modeling_with_content)) == 14922},
    {"check": "row_count_preserved_from_04", "value": len(modeling_with_content) == len(modeling_usage), "pass": len(modeling_with_content) == len(modeling_usage)},
    {"check": "membership_row_id_unique_preserved", "value": modeling_with_content["membership_row_id"].is_unique, "pass": modeling_with_content["membership_row_id"].is_unique},
    {"check": "content_feature_file_exists", "value": str(PATH_CONTENT_FEATURES), "pass": PATH_CONTENT_FEATURES.exists()},
    {"check": "modeling_with_content_file_exists", "value": str(PATH_MODELING_WITH_CONTENT), "pass": PATH_MODELING_WITH_CONTENT.exists()},
    {"check": "summary_json_output_exists", "value": str(PATH_CONTENT_SUMMARY_JSON), "pass": PATH_CONTENT_SUMMARY_JSON.exists()},
    {"check": "has_content_feature_rows_gt_zero", "value": int(modeling_with_content["has_content_feature"].sum()), "pass": int(modeling_with_content["has_content_feature"].sum()) > 0},
    {"check": "view_meta_rows_equals_obs_view_rows", "value": int(len(view_meta)), "pass": len(view_meta) == len(obs_view)},
    {"check": "usable_metadata_movie_count_gt_zero", "value": int(view_meta.loc[view_meta["use_for_content_features"] == 1, "MOVIE_NUM"].nunique()), "pass": int(view_meta.loc[view_meta["use_for_content_features"] == 1, "MOVIE_NUM"].nunique()) > 0},
])

final_check.to_csv(TABLES_DIR / "05_content_feature_final_checks.csv", index=False, encoding="utf-8-sig")
display(final_check)

if not final_check["pass"].all():
    failed = final_check.loc[~final_check["pass"]]
    raise AssertionError(f"05번 최종 검산 실패:\n{failed}")

print("05_content_feature_engineering passed final checks.")


,check,value,pass
0,project_root_is_repo_root,c:\Code\ott-churn-prediction,True
1,data_root_is_repo_data,c:\Code\ott-churn-prediction\_data,True
2,reports_dir_is_park_reports,c:\Code\ott-churn-prediction\park.ingyeom\reports,True
3,input_02_data_dir_is_reports_data_02,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
4,input_03_data_dir_is_reports_data_03,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
5,input_04_data_dir_is_reports_data_04,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
6,output_data_dir_under_reports_data_05,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
7,output_table_dir_under_reports_tables_05,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
8,only_expected_05_report_csvs,"[05_content_feature_attach_check.csv, 05_conte...",True
9,movie_meta_rows_is_14018,14018,True


05_content_feature_engineering passed final checks.


## 5-19. 05번 노트북 결론

05번 노트북의 최종 산출물은 `modeling_feature_table_with_content.csv`이다. 이 파일은 04번 사용 행동 피처 테이블에 영화 메타데이터 기반 콘텐츠 성향 피처를 붙인 모델링 후보 테이블이다.

다음 단계는 `06_significance_tests.ipynb`이다. 06번에서는 04번 행동 피처와 05번 콘텐츠 피처를 모두 포함해, 전체 고객, 100원딜 고객, 100원딜+2인, 100원딜+4인 등 주요 분석군별로 p-value, FDR 보정 p-value, 효과크기, 재구독률 차이를 계산한다.
